# 多头注意力

上一章，我们给自注意力加上了 $Q$、$K$、$V$ 角色分离机制。但是还有一个限制：所有词元共用同一套 $Q$、$K$、$V$。

就像一个人只会用一种视角看问题，无法理解同一个词在不同语境下可以有不同的含义。比如**苹果**这个词：

* 在**吃苹果**中，它是水果；
* 在**苹果手机**中，它是品牌。

目前的模型只能用同一种方式去理解它，无法同时捕捉这两种不同的含义。

## 多头注意力

解决办法是**多头注意力**（Multi-Head Attention）：把词元的嵌入向量切分成若干份，称为**头**（Head）；给每份生成单独的一套 $Q$、$K$、$V$ 线性投影；然后各自发展出独立的注意力，最后再拼合到一起：

* 每个**头**有自己的 $Q$、$K$、$V$ 线性投影，关注不同的特征子空间；
* 每个头独立发展，关注到不同的数据逻辑；
* 多头并行工作，最后汇总，模型就能从多个角度理解上下文。

### 切分与合并

假设嵌入尺寸是 64，头数是 2，那么：

* 每个头的嵌入尺寸是 64 / 2 = 32；
* 输入格式从 `(批大小, 上下文尺寸, 64)` 变成 `(批大小, 上下文尺寸, 2, 32)`；
* 转置成 `(批大小, 2, 上下文尺寸, 32)`，这样每个头可以独立计算注意力；
* 计算完后转置回来 `(批大小, 上下文尺寸, 2, 32)`，再拼接回 `(批大小, 上下文尺寸, 64)`。

### 缩放因子

每个头的嵌入尺寸是 `EMBEDDING_SIZE / HEADS`，所以缩放因子 $1 / \sqrt{d}$ 里的 $d$ 现在是 EMBEDDING_SIZE / HEADS。

### 参数量

多头注意力的参数量和单头注意力完全一样：$Q$、$K$、$V$ 三个线性投影的权重矩阵依然是 `(嵌入尺寸, 嵌入尺寸)`。区别只在于计算时把向量切分成多份，每份独立计算。

``💡 计算多头注意力不是多个注意力的堆叠，而是同一个注意力内部扩展维度后的并行计算。``

In [1]:
from pathlib import Path

import requests

from src.core import *

In [2]:
np.random.seed(42)

## 数据集

### 字符编码数据集

In [3]:
class CharDataset(Dataset):

    def __init__(self, filename, batch_size=1, context_size=64, stride=None, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.stride = stride if stride is not None else context_size // 2
        self.split = split
        super().__init__(batch_size)

    def load(self):
        with open(self.filename, encoding="utf-8") as f:
            text = f.read()

        self.vocab = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.tokens = self.encode(text)

        split = int(len(self.tokens) * self.split)
        self.train_data = self._pack(self.tokens[:split])
        self.test_data = self._pack(self.tokens[split:])

    def _pack(self, tokens):
        x, y = [], []
        for i in range(0, len(tokens) - self.context_size - 1, self.stride):
            x.append(tokens[i: i + self.context_size])
            y.append(tokens[i + 1: i + self.context_size + 1])
        return x, y

    def encode(self, symbols):
        return [self.stoi[s] for s in symbols]

    def decode(self, tokens):
        return "".join(self.itos[t] for t in tokens)

## 下三角矩阵层

In [4]:
class Tril(Layer):

    def __init__(self, value=-1e9):
        super().__init__()
        self.value = value

    def forward(self, x: Tensor):
        keep = np.tril(np.ones(x.shape[-2:]))
        p = Tensor(np.where(keep, x.data, self.value))

        def gradient_fn():
            x.grad += p.grad * keep

        return p.attach(gradient_fn, {x})

## 激活函数

### 高斯误差线性单元激活函数（GELU）

In [5]:
class GELU(Layer):

    def __init__(self):
        super().__init__()
        self.c = np.sqrt(2.0 / np.pi)

    def forward(self, x: Tensor):
        tanh = np.tanh(self.c * (x.data + 0.044715 * x.data ** 3))
        a = Tensor(0.5 * x.data * (1.0 + tanh))

        def gradient_fn():
            grad = 0.5 * (1.0 + tanh) + 0.5 * x.data * (1.0 - tanh ** 2) * self.c * (1.0 + 3.0 * 0.044715 * x.data ** 2)
            x.grad += a.grad * grad

        return a.attach(gradient_fn, parents={x})

## 损失函数

### GPT 交叉熵损失函数

In [6]:
class GPTLoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        flat_softmax = softmax.reshape(-1, softmax.shape[-1])
        flat_y = y.data.reshape(-1).astype(np.int64)
        n = len(flat_y)
        rows = np.arange(n)

        log = np.log(np.clip(flat_softmax[rows, flat_y], 1e-10, 1))
        ce = Tensor(0 - np.sum(log) / n)

        def gradient_fn():
            flat_grad = flat_softmax.copy()
            flat_grad[rows, flat_y] -= 1
            p.grad += ce.grad * flat_grad.reshape(softmax.shape) / n

        return ce.attach(gradient_fn, parents={p})

## 模型

### GPT 嵌入层

In [7]:
class GPTEmbedding(Composite):

    def __init__(self, vocab_size, context_size, embedding_size):
        self.embedding = Embedding(vocab_size, embedding_size)
        self.positional_embedding = Embedding(context_size, embedding_size)

        super().__init__([self.embedding,
                          self.positional_embedding])

    def forward(self, x: Tensor):
        token = self.embedding(x)
        position = self.positional_embedding(Tensor(range(x.shape[1])))
        return token + position

### GPT 注意力层

我们在 GPT 注意力层添加多头支持。

前向传播多了**切分**和**合并**两个步骤：

* **生成 $Q$、$K$、$V$**：通过三个线性层，分别得到查、键、值；
* **切分头部**：把嵌入向量拆分成若干份；
* **计算注意力**：每个头部独立计算注意力分数矩阵、缩放、掩码、计算权重、加权求和；
* **合并头部**：将多头注意力拼接汇总。

注意力在切分和合并时格式的变化：

$$
\text{切分}: (\text{批}, \text{上下文}, \text{嵌入}) \xrightarrow{\text{切分}} (\text{批}, \text{上下文}, \text{头数}, \text{每头嵌入}) \xrightarrow{\text{转置}} (\text{批}, \text{头数}, \text{上下文}, \text{每头嵌入})
$$

$$
\text{合并}: (\text{批}, \text{头数}, \text{上下文}, \text{每头嵌入}) \xrightarrow{\text{转置}} (\text{批}, \text{上下文}, \text{头数}, \text{每头嵌入}) \xrightarrow{\text{合并}} (\text{批}, \text{上下文}, \text{嵌入})
$$

因此，多头机制只在 GPT 注意力层内部处理。输出结果的格式没有变化，后续步骤不需要修改。

In [8]:
class GPTAttention(Composite):

    def __init__(self, embedding_size, heads=1):
        self.embedding_size = embedding_size
        self.heads = heads

        self.query = Linear(embedding_size, embedding_size)
        self.key = Linear(embedding_size, embedding_size)
        self.value = Linear(embedding_size, embedding_size)
        self.mask = Tril()
        self.softmax = Softmax()
        self.output = Linear(embedding_size, embedding_size)

        super().__init__([self.query,
                          self.key,
                          self.value,
                          self.mask,
                          self.softmax,
                          self.output])

    def forward(self, x: Tensor):
        # 每头嵌入尺寸
        head_embedding_size = self.embedding_size // self.heads
        # 多头格式
        multi_head_shape = (-1, x.shape[1], self.heads, head_embedding_size)
        # 合并后格式
        shape = (-1, x.shape[1], self.embedding_size)

        # （多头）查
        query = self.query(x).reshape(multi_head_shape).transpose((0, 2, 1, 3))
        # （多头）键，提前转置
        key = self.key(x).reshape(multi_head_shape).transpose((0, 2, 3, 1))
        # （多头）值
        value = self.value(x).reshape(multi_head_shape).transpose((0, 2, 1, 3))
        # 缩放因子
        scale = Tensor(np.array(1.0 / np.sqrt(head_embedding_size)))
        # 注意力分数矩阵（查、键）
        scores = query @ key * scale
        # 因果注意力权重
        weights = self.softmax(self.mask(scores))
        # （合并后）缩放点积注意力（值）
        return self.output((weights @ value).transpose((0, 2, 1, 3)).reshape(shape))

### GPT 前馈网络层

In [9]:
class GPTFeedForward(Composite):

    def __init__(self, embedding_size):
        self.input = Linear(embedding_size, embedding_size * 4)
        self.gelu = GELU()
        self.output = Linear(embedding_size * 4, embedding_size)

        super().__init__([self.input,
                          self.gelu,
                          self.output])

    def forward(self, x: Tensor):
        h = self.gelu(self.input(x))
        return self.output(h)

### GPT 转换器模块

创建时需要传入 heads 参数。

In [10]:
class GPTTransformer(Composite):

    def __init__(self, embedding_size, heads):
        self.attention = GPTAttention(embedding_size, heads)
        self.feed_forward = GPTFeedForward(embedding_size)

        super().__init__([self.attention,
                          self.feed_forward])

    def forward(self, x: Tensor):
        x = self.attention(x)
        return self.feed_forward(x)

### GPT 输出层

In [11]:
class GPTOutput(Composite):

    def __init__(self, embedding_size, vocab_size):
        self.output = Linear(embedding_size, vocab_size)

        super().__init__([self.output])

    def forward(self, x: Tensor):
        return self.output(x)

### GPT 模型

创建时需要传入 heads 参数。

In [12]:
class GPT(Composite):

    def __init__(self, vocab_size, context_size, embedding_size, heads, blocks):
        self.embedding = GPTEmbedding(vocab_size, context_size, embedding_size)
        self.transformers = [GPTTransformer(embedding_size, heads) for _ in range(blocks)]
        self.output = GPTOutput(embedding_size, vocab_size)

        super().__init__([self.embedding] + self.transformers + [self.output])

    def forward(self, x: Tensor):
        x = self.embedding(x)
        for layer in self.transformers:
            x = layer(x)
        return self.output(x)

## GPT 训练器

In [13]:
class GPTTrainer(Trainer):

    def train(self, dataset, epochs, scheduler=None):
        dataset.train()
        self.model.train()

        steps = 0
        for epoch in range(epochs):
            for i in range(len(dataset)):
                if scheduler is not None:
                    self.optimizer.lr = scheduler.step(steps)

                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.model(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.clip_grad_norm()
                self.optimizer.step()
                steps += 1

    def test(self, dataset):
        dataset.eval()
        self.model.eval()

        predictions = []
        total_loss = 0.0
        with Tensor.no_grad():
            for i in range(len(dataset)):
                feature, label = dataset[i]
                prediction = self.model(feature)
                loss = self.loss_fn(prediction, label)
                predictions.append(prediction)
                total_loss += float(loss.data)

        loss = total_loss / len(dataset)
        dataset.train()
        self.model.train()
        return predictions, loss

    def generate(self, dataset, prompt, steps=256):
        self.model.eval()
        tokens = dataset.encode(prompt)

        with Tensor.no_grad():
            for _ in range(steps):
                feature = Tensor([tokens[-dataset.context_size:]])
                logits = self.model(feature)

                last_logits = logits.data[0, -1]
                exp = np.exp(last_logits - np.max(last_logits))
                probs = exp / np.sum(exp)
                token = np.random.choice(len(probs), p=probs)
                tokens.append(token)

        return dataset.decode(tokens)

## 超参数

### 学习率

In [14]:
LEARNING_RATE = 0.001

### 批大小

In [15]:
BATCH_SIZE = 4

### 轮数

In [16]:
EPOCHS = 2

### 上下文尺寸

In [17]:
CONTEXT_SIZE = 32

### 嵌入尺寸

In [18]:
EMBEDDING_SIZE = 64

### 头数

这是定义多头注意力中**头数**的超参数。

In [19]:
HEADS = 2

### 转换器模块数

In [20]:
BLOCKS = 2

## 建模

建模时，需要额外传入 HEADS 作为模型参数。

In [21]:
DATA_FILE = "../../../../data/tinyshakespeare.txt"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(model.parameters, lr=LEARNING_RATE)
trainer = GPTTrainer(model, loss_fn, optimizer)

## 训练

In [22]:
scheduler = WarmupCosineScheduler(LEARNING_RATE, EPOCHS * len(dataset), 100, LEARNING_RATE / 10)
trainer.train(dataset, EPOCHS, scheduler)

## 评估

In [23]:
prediction, loss = trainer.test(dataset)
print(f'prediction:\t{len(prediction)} steps, each {prediction[0].data.shape}\nloss:\t{loss}')

prediction:	1742 steps, each (4, 32, 65)
loss:	1.8594153204892176


## 推理

In [24]:
print(trainer.generate(dataset, prompt="ROMEO:"))

ROMEO:
Where as her?

HORTESRETINSA:
Marh's the tranten, sir; aft a with roonoous look him word.

HORTENSI:
What, and an he the lady not; fault's it thee grat a in life,
Und pertinact
Beself?

KATHINO:
Thetrher.
Came.

HORTENSIO:
Hither, silvely? My Lordiet, at 


本章，我们给注意力加上了**多头机制**：把嵌入向量切分成若干份，每份拥有独立的 $Q$、$K$、$V$ 线性投影，独立做注意力，最终拼接汇总。模型因此可以从多个角度理解上下文。

至此，我们实现了完整的**自注意力机制**，一步步搭建出 GPT 模型的核心组件：

* **自注意力**：词元之间看到彼此；
* **因果注意力**：屏蔽未来，防止偷看答案；
* **加权注意力**：$Q$、$K$、$V$ 角色分离；
* **多头注意力**：多角度并行理解上下文。

## 课后练习

试着把 HEADS 从 2 改成 4 或者 8，观察训练速度和效果有什么变化？头数是不是越多越好？